# 02 — WoE binning and Information Value

**Goal:** bin every candidate feature, convert bins to Weight of Evidence, and rank
features by Information Value (IV).

$$\text{WoE}_i = \ln\frac{\%\text{good}_i}{\%\text{bad}_i} \qquad
\text{IV} = \sum_i (\%\text{good}_i - \%\text{bad}_i)\,\text{WoE}_i$$

Positive WoE = safer than average. Bins are fitted on **train only**.

In [ ]:
import os, sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pdmodel import config as C
pd.set_option("display.float_format", "{:.4f}".format)
PROC = ROOT / "data" / "processed"
from pdmodel import plots
from pdmodel.woe import WoEBinner

In [ ]:
S = pd.read_pickle(PROC / "samples.pkl")
train = S["train"]

## 1. Fit the binner
For each numeric feature (see `src/pdmodel/woe.py`):
1. start from 10 quantile bins,
2. merge any bin with < 5% of the sample (small bins = noisy WoE),
3. merge adjacent bins until the bad rate is **monotonic** (always rising or always falling).

Step 3 is a business choice: it guarantees, for example, that a higher FICO can never
get *fewer* points, which credit officers and regulators expect.

In [ ]:
binner = WoEBinner().fit(train, train["default"], C.NUMERIC_FEATURES, C.CATEGORICAL_FEATURES)
iv = binner.iv_table()
iv

IV rule of thumb: < 0.02 useless · 0.02–0.1 weak · 0.1–0.3 medium · 0.3–0.5 strong · > 0.5 suspicious (leakage?).

## 2. Look at the bins of the strongest features

In [ ]:
for f in iv.index[:4]:
    plots.woe_plot(binner.bins[f].table); plt.show()

In [ ]:
binner.bins[iv.index[0]].table

## 3. Categorical features
Rare categories (< 1%) are pooled into `OTHER` so their WoE isn't driven by a handful of loans.

In [ ]:
binner.bins["purpose"].table

In [ ]:
pd.to_pickle(binner, PROC / "binner.pkl")

## ✍️ Try it
- Re-fit with `WoEBinner(monotonic=False)`. Which features change most? Is the extra IV worth the loss of explainability?
- Change `MIN_BIN_FRAC` to 0.02. What happens to IV on train vs. stability later?

## ✍️ Interview prep
1. Why does WoE make logistic regression a good fit for scorecards?
2. Why fit bins on train only?
3. A feature has IV = 0.9. What do you do?